# Build index — Kaggle **2× T4** — dùng clean.jsonl ĐÃ VÁ (one-click)

**KHÁC `build_index_kaggle_clean.ipynb`:** notebook này **KHÔNG fix data trên Kaggle nữa**. Data đã vá xong ở local (`legal_articles.clean.jsonl`: Lao động 45/2019 VI 220 Điều, Đấu thầu 22/2023 95 Điều, EN-residue=0, 148.106 row). Cell 4 chỉ **swap clean→main + verify**, KHÔNG re-crawl HF (tránh poison như lần trước).

## Chuẩn bị dataset Kaggle (BẮT BUỘC)
Cập nhật dataset `r2ai-pipeline` để chứa **`data/legal_articles.clean.jsonl`** (bản vừa vá ở local) cùng `src/`, `scripts/`, `config.yaml`. KHÔNG cần `corpus.json` (không vá data nữa).

## Settings panel bên phải
- Accelerator → **GPU T4 ×2**
- Internet → **On** (để pip cài lib + tải model embedding; KHÔNG crawl luật)
- Add Data → dataset `r2ai-pipeline`

Output: `/kaggle/working/r2ai_index_clean.zip` (index/bm25.pkl + dense.pkl + jsonl + manifest).

In [ ]:
# 1) GPU phải hiện 2 dòng Tesla T4
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 2) Cài lib (KHÔNG cần 'datasets' — không crawl HF)
!pip install -q -U sentence-transformers rank_bm25 pyvi scikit-learn pyyaml

In [ ]:
# 3) Copy mã nguồn sang vùng ghi được + tìm clean.jsonl
import os, glob, shutil, pathlib
src = None
for cfg in glob.glob('/kaggle/input/*/**/config.yaml', recursive=True):
    p = pathlib.Path(cfg).parent
    if (p/'src').is_dir() and (p/'scripts').is_dir():
        src = p; break
assert src, 'Khong thay r2ai_pipeline trong Add Data.'
PROJ = pathlib.Path('/kaggle/working/r2ai_pipeline')
if PROJ.exists(): shutil.rmtree(PROJ)
shutil.copytree(src, PROJ); os.chdir(PROJ)
assert (PROJ/'data/legal_articles.clean.jsonl').exists(), \
    'THIEU data/legal_articles.clean.jsonl trong dataset! Phai up ban da va o local len.'
sz = (PROJ/'data/legal_articles.clean.jsonl').stat().st_size / 1e6
print(f'CWD: {os.getcwd()} | clean.jsonl = {sz:.0f} MB')

In [ ]:
# 4) === SWAP clean -> main (build script doc cung data/legal_articles.jsonl) + VERIFY ===
#    KHONG fix data nua. Chi xac nhan ban clean dung truoc khi build.
import shutil, json, re
from collections import Counter

if pathlib.Path('data/legal_articles.jsonl').exists():
    shutil.copy('data/legal_articles.jsonl','data/legal_articles.en_orig.jsonl')
shutil.copy('data/legal_articles.clean.jsonl','data/legal_articles.jsonl')

CORE5 = {'36/2005/QH11','45/2019/QH14','04/2017/QH14','14/2008/QH12','13/2008/QH12'}
en = Counter(); rows = 0; labor = 0; dauthau = set()
for line in open('data/legal_articles.jsonl', encoding='utf-8'):
    if not line.strip(): continue
    r = json.loads(line); rows += 1
    d = str(r.get('doc_num',''))
    if d in CORE5 and re.match(r'\s*\**Article\s+\d+', r['text']):
        en[d] += 1
    if d == '45/2019/QH14': labor += 1
    if d == '22/2023/QH15':
        m = re.search(r'(\d+)', str(r.get('article_number','')))
        if m: dauthau.add(int(m.group(1)))
print(f'rows={rows:,} | Lao dong 45/2019 rows={labor} | Dau thau 22/2023 dieu={len(dauthau)}')
print(f'Dau thau missing 1..96 = {sorted(set(range(1,97))-dauthau)} (chi nen con [71])')
print('English Dieu con lai / luat loi (mong doi 0 ca 5):', dict(en) or 'TAT CA = 0')
assert sum(en.values()) == 0, 'Con Dieu tieng Anh -> clean.jsonl chua dung. DUNG build.'
assert labor == 220, f'Lao dong phai 220 row, dang {labor}.'
print('OK -> san sang build.')

In [ ]:
# 5) Cau hinh build: st + fp16 + cuda
import re, pathlib
cfgp = pathlib.Path('config.yaml'); t = cfgp.read_text(encoding='utf-8')
t = re.sub(r'dense_backend:\s*\w+','dense_backend: st', t)
t = re.sub(r'dense_fp16:\s*\w+','dense_fp16: true', t)
t = re.sub(r'device:\s*\w+','device: cuda', t)
cfgp.write_text(t, encoding='utf-8')
!grep -E 'dense_backend|dense_model|dense_fp16|device' config.yaml

In [ ]:
# 6) Build multi-GPU (OOM thi giam --batch-size 64). Tao lai CA bm25.pkl + dense.pkl
!python scripts/build_index_kaggle_from_jsonl.py --batch-size 128 --devices cuda:0,cuda:1

In [ ]:
# 7) Kiem tra output
!ls -lh data/index/

In [ ]:
# 8) Dong goi: index + jsonl + manifest
import json, hashlib, pathlib
def md5(p):
    h=hashlib.md5()
    with open(p,'rb') as f:
        [h.update(b) for b in iter(lambda:f.read(1<<20), b'')]
    return h.hexdigest()
man={'index':{f:md5('data/index/'+f) for f in ['bm25.pkl','dense.pkl']},
     'jsonl_rows':sum(1 for _ in open('data/legal_articles.jsonl'))}
pathlib.Path('data/index/MANIFEST.json').write_text(json.dumps(man,indent=2,ensure_ascii=False))
print(man)
!cp data/legal_articles.jsonl data/index/legal_articles.clean.jsonl
!cd data && zip -r /kaggle/working/r2ai_index_clean.zip index/ >/dev/null && ls -lh /kaggle/working/r2ai_index_clean.zip

## Sau khi chay
1. Tai `r2ai_index_clean.zip` tu tab **Output**.
2. Cap nhat dataset inference bang `index/` moi (bm25.pkl + dense.pkl).
3. Chay `kaggle_inference.ipynb` voi config LONG (1420): `max_k=8, rel_threshold=0.3, rel_score_transform=none`. **TAT Cell 3b auto-tune dev** (dev khong dai dien public — da lam 1545 tut diem).
4. Verify offline truoc khi nop (con it luot, deadline 30/06).